# DynamicAuc Simulator Tutorial

This notebook provides a complete walkthrough of the `dynamicauc` simulator. We will cover:

1.  **Basic Simulation**: Running a campaign with a single, simple policy.
2.  **Benchmarking Policies**: Comparing multiple baseline policies in a statistically robust way.
3.  **Integrating Reinforcement Learning**: Training an RL agent and benchmarking it against the baselines.
4.  **Visualizing Results**: Plotting the performance of different policies.
5.  **Command-Line Usage**: Using Hydra to run simulations from the terminal.

## Setup and Installation

First, ensure `dynamicauc` is installed. For this tutorial, we will also need `stable-baselines3` to train our RL agent and `seaborn` for plotting.

In [ ]:
!pip install stable-baselines3[extra] seaborn

  Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (1.7 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 10.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 15.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 266.6/530.7 MB 24.8 MB/s eta 0:00:11

In [ ]:
import dynamicauc as dac
from dynamicauc.simulator import Simulator
from dynamicauc.policies import FiPlaPolicy, ConstantPolicy, RandomPolicy

import gymnasium as gym
from stable_baselines3 import PPO

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Part 1: Basic Simulation (Single Policy)

The core components of a simulation are a `Scenario` (the environment's rules) and a `BiddingPolicy` (the agent's strategy). We can create a scenario from a built-in preset.

In [ ]:
# 1. Create a scenario from a preset
scenario = dac.make("saturation-first-price-uniform")

# 2. Instantiate a policy
policy = FiPlaPolicy()

# 3. Configure and run the simulator
sim = Simulator(scenario, policy, campaign_cfg={
    "horizon": 10.0,
    "n_runs": 10,
    "seed": 42,
})

auctions_df, displays_df, conversions_df, summary_df = sim.run()

print("--- Simulation Summary (FiPla Policy) ---")
summary_df.head()

## Part 2: Benchmarking Multiple Policies

The `Simulator` is designed to easily compare multiple policies. By passing a dictionary of policies, the simulator will automatically run all of them in a **paired experiment**. This means for each `run_id`, every policy faces the exact same sequence of auction arrivals and competitor bids, leading to a much fairer and more statistically robust comparison.

In [ ]:
# 1. Define a dictionary of policies to compare
policies = {
    "FiPla": FiPlaPolicy(),
    "Constant(0.5)": ConstantPolicy(0.5),
    "Random(2.0)": RandomPolicy(2.0),
}

# 2. Run the simulator with the policy dictionary
multi_policy_sim = Simulator(scenario, policies, campaign_cfg={
    "horizon": 10.0,
    "n_runs": 50, # More runs for better statistics
    "seed": 123,
})

_, _, _, multi_summary_df = multi_policy_sim.run()

# The output summary now includes a 'policy' column
print("--- Multi-Policy Simulation Summary ---")
multi_summary_df.head()

We can now compute aggregate statistics for each policy.

In [ ]:
baseline_results = multi_summary_df.groupby("policy").agg(
    mean_conversions=("n_conversions", "mean"),
    std_conversions=("n_conversions", "std"),
    mean_spend=("total_spend", "mean")
).reset_index()

print("--- Aggregate Performance of Baseline Policies ---")
baseline_results

## Part 3: Integrating a Trained Reinforcement Learning Agent

A key feature of `dynamicauc` is its `Gymnasium` compatibility, allowing you to train agents with standard RL libraries. Here's the full workflow to train a PPO agent and add it to our benchmark.

### Step 3.1: Train the RL Agent

We create a `DynamicAuction-v0` environment and train a `PPO` agent from `stable-baselines3`. The agent will learn a policy by interacting directly with the environment.

In [ ]:
# 1. Create the Gym-compatible environment
env = gym.make("DynamicAuction-v0", scenario=scenario)

# 2. Instantiate and train the PPO agent
# The 'MultiInputPolicy' is required because our observation is a dictionary
model = PPO("MultiInputPolicy", env, verbose=0, seed=42)
print("Training PPO agent... (this may take a minute)")
model.learn(total_timesteps=20000) # Use more timesteps for better performance
print("Training complete.")

### Step 3.2: Add the Trained Agent to the Benchmark

The `Simulator` can accept any `callable` that takes an observation dictionary and returns a bid. This makes it trivial to integrate our trained model.

We create a simple `lambda` function that wraps the model's `predict` method and add it to our dictionary of policies.

In [ ]:
# 1. Create a callable function from the trained model
# model.predict() returns (action, state), so we take the first element.
rl_policy_fn = lambda obs: model.predict(obs, deterministic=True)[0]

# 2. Add the RL agent to our policy dictionary
policies["PPO_Agent"] = rl_policy_fn

print("Policies to be benchmarked:", list(policies.keys()))

### Step 3.3: Run the Full Comparison

Now we run the simulator one last time with all policies, including our new PPO agent.

In [ ]:
full_sim = Simulator(scenario, policies, campaign_cfg={
    "horizon": 10.0,
    "n_runs": 100, # Use a high number of runs for a reliable comparison
    "seed": 456,
})

_, _, _, final_summary_df = full_sim.run()

final_results = final_summary_df.groupby("policy").agg(
    mean_conversions=("n_conversions", "mean"),
    std_conversions=("n_conversions", "std"),
    mean_spend=("total_spend", "mean")
).reset_index()

print("--- Final Benchmark Results ---")
final_results

## Part 4: Visualizing the Results

A plot is often the best way to compare performance. We'll use `seaborn` to create bar plots for the average number of conversions and the average total spend.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100)

# Plot 1: Mean Conversions
sns.barplot(data=final_results, x="policy", y="mean_conversions", ax=axes[0])
axes[0].set_title("Average Conversions per Campaign", fontweight="bold")
axes[0].set_ylabel("Mean Conversions")
axes[0].set_xlabel("Policy")
axes[0].tick_params(axis='x', rotation=45)

# Plot 2: Mean Spend
sns.barplot(data=final_results, x="policy", y="mean_spend", ax=axes[1])
axes[1].set_title("Average Spend per Campaign", fontweight="bold")
axes[1].set_ylabel("Mean Total Spend")
axes[1].set_xlabel("Policy")
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Part 5: Using Hydra for Command-Line Simulations

For large-scale experiments, you can use the provided `dynamicauc/main.py` script, which is configured with Hydra. This allows you to easily override any parameter from the command line.

The following command runs the default configuration (`conf/config.yaml`), which uses the `fipla` policy.

In [ ]:
!python ../dynamicauc/main.py campaign.n_runs=5

You can easily swap out components. For example, to run with the `constant` policy and a different competition model, you would run:

```bash
python ../dynamicauc/main.py policy=constant competition=lognormal campaign.n_runs=5
```

This is ideal for running sweeps and generating data for research papers.